<a href="https://colab.research.google.com/github/VictorVega03/Programacion-de-sistemas-base-2/blob/main/Copia_de_Pr%C3%A1ctica_Tabla_de_s%C3%ADmbolos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación de Sistemas de Base II - Práctica: Elementos de la Programación de Sistemas, Máquinas de Pila y Ámbitos de Tabla de Símbolos
  
**Asignatura:** Programación de Sistemas de Base II (9.º Semestre)  
**Unidad II:** Elementos de la Programación de Sistemas  
**Tiempo estimado:** 3 horas (Asíncrono / Guiado)  
**Repositorio Base / Entrega:** Integración con Moodle / GitHub

## 1. Objetivos de Aprendizaje    
1. Simular el comportamiento de una máquina de pila abstracta para la evaluación y generación de instrucciones.  
2. Construir e inspeccionar una Tabla de Símbolos avanzada con soporte para bloques anidados y manejo de ámbitos (Scopes).  
3. Comprender el mecanismo de traducción de expresiones simples y el rol de los ensambladores y cargadores en la cadena de desarrollo de sistemas.

## 2. Conexión Teórica (NotebookLM)    
**Antes de continuar:** Accede al espacio de **NotebookLM del Curso** y realiza las siguientes preguntas de verificación:    
> - *¿Qué diferencia a un ensamblador de una pasada de uno de dos pasadas?*    
> - *¿Cómo evalúa una máquina de pila la expresión posfija `a b + c *`?*    
> - *¿Qué representa una entrada en la Tabla de Símbolos dentro de un entorno de ejecución?*    


## 3. Sección Práctica 1: Simulación de una Máquina de Pila Abstracta (Stack Machine)  
Este código simula paso a paso la mecánica de un analizador ascendente (LR) con Pila Semántica y evaluación On-the-Fly.  
Se muestra el apilado/desapilado de tuplas semánticas y la emisión inmediata de Código de Tres Direcciones (3AC) conforme ocurren las reducciones, sin necesidad de construir un árbol sintáctico explícito en memoria.

### [CÓDIGO 1.1] Implementación del Emulador de Máquina de Pila   

In [7]:
# ==============================================================================
# PROGRAMACIÓN DE SISTEMAS DE BASE II
# SIMULACIÓN DE PILA SEMÁNTICA Y EVALUACIÓN "ON-THE-FLY" (AL VUELO) EN PYTHON
# ==============================================================================

class ElementoPila:
    """Estructura de la tupla/registro de la Pila Semántica: (estado, simbolo, atributo)"""
    def __init__(self, estado, simbolo, atributo=None):
        self.estado = estado
        self.simbolo = simbolo
        self.atributo = atributo  # Almacena tipos, valores o temporales

    def __repr__(self):
        val_str = f", attr={self.atributo}" if self.atributo is not None else ""
        return f"[{self.estado}: {self.simbolo}{val_str}]"


class SimulacionPilaSemantica:
    """
    Simula un analizador ascendente (LR) con Pila Semántica paralela
    que realiza la evaluación 'On-the-Fly' (al vuelo) generando Código de
    Tres Direcciones (3AC) sin construir un árbol sintáctico explícito en memoria.
    """
    def __init__(self):
        self.pila = [ElementoPila(0, '$')]  # Estado inicial 0 con centinela $
        self.contador_temporales = 0
        self.codigo_intermedio = []

    def nuevo_temporal(self):
        self.contador_temporales += 1
        return f"t{self.contador_temporales}"

    def emitir(self, instruccion):
        self.codigo_intermedio.append(instruccion)
        print(f"   ⚡ [ON-THE-FLY EMISIÓN]: {instruccion}")

    def mostrar_pila(self):
        pila_str = " ".join([str(elem) for elem in self.pila])
        print(f"PILA: {pila_str}")

    def desplazar(self, nuevo_estado, simbolo, atributo=None):
        """Operación Shift: Apila el estado, el símbolo y su atributo léxico ($1, $2...)"""
        elem = ElementoPila(nuevo_estado, simbolo, atributo)
        self.pila.append(elem)
        print(f"\n---> DESPLAZAR (Shift): Token '{simbolo}' (Atributo: {atributo}) -> Estado {nuevo_estado}")
        self.mostrar_pila()

    def reducir(self, num_regla, antecedente, num_simbolos_cuerpo, accion_semantica_fn, nuevo_estado_ir_a):
        """
        Operación Reduce:
        1. Desapila los 'num_simbolos_cuerpo' elementos de la regla.
        2. Accede a sus atributos ($1, $2, ..., $n).
        3. Ejecuta la acción semántica On-the-Fly ($$).
        4. Apila el antecedente con su nuevo atributo ($$) y el estado IR_A.
        """
        print(f"\n<--- REDUCIR (Reduce R{num_regla}): {antecedente} -> ({num_simbolos_cuerpo} símbolos)")

        # Extraer operandos del cuerpo de la regla desde la pila
        elementos_extraidos = []
        for _ in range(num_simbolos_cuerpo):
            elementos_extraidos.insert(0, self.pila.pop())

        # Los atributos extraídos corresponden a $1, $2, ..., $n
        atributos_cuerpo = [elem.atributo for elem in elementos_extraidos]
        print(f"   Atributos desapilados ($1..${num_simbolos_cuerpo}): {atributos_cuerpo}")

        # Ejecución de la acción semántica On-the-fly para calcular $$
        atributo_sintetizado = accion_semantica_fn(atributos_cuerpo)

        # Apilar antecedente con el estado indicado por la tabla IR_A
        self.pila.append(ElementoPila(nuevo_estado_ir_a, antecedente, atributo_sintetizado))
        print(f"   Resultado sintetizado ($$ = {atributo_sintetizado}) apilado en estado {nuevo_estado_ir_a}")
        self.mostrar_pila()

# ==============================================================================
# EJECUCIÓN PASO A PASO DE LA SIMULACIÓN
# Cadena de entrada: "a + b * 5"
# Gramática y Reglas Semánticas:
#  (1) E -> E + T   { $$ = nuevo_temp(); emitir($$ = $1 + $3); }
#  (2) E -> T       { $$ = $1; }
#  (3) T -> T * F   { $$ = nuevo_temp(); emitir($$ = $1 * $3); }
#  (4) T -> F       { $$ = $1; }
#  (5) F -> id      { $$ = $1.nombre; }
#  (6) F -> num     { $$ = $1.valor; }
# ==============================================================================

if __name__ == "__main__":
    sim = SimulacionPilaSemantica()
    print("=" * 70)
    print("SIMULACIÓN DE LA PILA SEMÁNTICA Y TRADUCCIÓN ON-THE-FLY DE 'a + b * 5'")
    print("=" * 70)
    sim.mostrar_pila()

    # 1. Shift 'id' (a)
    sim.desplazar(nuevo_estado=3, simbolo='id', atributo='a')

    # 2. Reduce (5) F -> id
    sim.reducir(
        num_regla=5, antecedente='F', num_simbolos_cuerpo=1,
        accion_semantica_fn=lambda attrs: attrs[0],  # $$ = $1 ('a')
        nuevo_estado_ir_a=2
    )

    # 3. Reduce (4) T -> F
    sim.reducir(
        num_regla=4, antecedente='T', num_simbolos_cuerpo=1,
        accion_semantica_fn=lambda attrs: attrs[0],  # $$ = $1 ('a')
        nuevo_estado_ir_a=1
    )

    # 4. Reduce (2) E -> T
    sim.reducir(
        num_regla=2, antecedente='E', num_simbolos_cuerpo=1,
        accion_semantica_fn=lambda attrs: attrs[0],  # $$ = $1 ('a')
        nuevo_estado_ir_a=4
    )

    # 5. Shift '+'
    sim.desplazar(nuevo_estado=6, simbolo='+')

    # 6. Shift 'id' (b)
    sim.desplazar(nuevo_estado=3, simbolo='id', atributo='b')

    # 7. Reduce (5) F -> id
    sim.reducir(
        num_regla=5, antecedente='F', num_simbolos_cuerpo=1,
        accion_semantica_fn=lambda attrs: attrs[0],  # $$ = $1 ('b')
        nuevo_estado_ir_a=2
    )

    # 8. Reduce (4) T -> F
    sim.reducir(
        num_regla=4, antecedente='T', num_simbolos_cuerpo=1,
        accion_semantica_fn=lambda attrs: attrs[0],  # $$ = $1 ('b')
        nuevo_estado_ir_a=7
    )

    # 9. Shift '*'
    sim.desplazar(nuevo_estado=8, simbolo='*')

    # 10. Shift 'num' (5)
    sim.desplazar(nuevo_estado=5, simbolo='num', atributo='5')

    # 11. Reduce (6) F -> num
    sim.reducir(
        num_regla=6, antecedente='F', num_simbolos_cuerpo=1,
        accion_semantica_fn=lambda attrs: attrs[0],  # $$ = $1 ('5')
        nuevo_estado_ir_a=9
    )

    # 12. Reduce (3) T -> T * F (Acción con emisión de código On-The-Fly)
    def accion_multiplicacion(attrs):
        t = sim.nuevo_temporal()
        sim.emitir(f"{t} = {attrs[0]} * {attrs[2]}")
        return t

    sim.reducir(
        num_regla=3, antecedente='T', num_simbolos_cuerpo=3,
        accion_semantica_fn=accion_multiplicacion,
        nuevo_estado_ir_a=7
    )

    # 13. Reduce (1) E -> E + T (Acción con emisión de código On-The-Fly)
    def accion_suma(attrs):
        t = sim.nuevo_temporal()
        sim.emitir(f"{t} = {attrs[0]} + {attrs[2]}")
        return t

    sim.reducir(
        num_regla=1, antecedente='E', num_simbolos_cuerpo=3,
        accion_semantica_fn=accion_suma,
        nuevo_estado_ir_a=4
    )

    print("\n" + "=" * 70)
    print("CÓDIGO DE TRES DIRECCIONES GENERADO ON-THE-FLY:")
    print("=" * 70)
    for i, inst in enumerate(sim.codigo_intermedio, 1):
        print(f" {i}. {inst}")
    print("=" * 70)

SIMULACIÓN DE LA PILA SEMÁNTICA Y TRADUCCIÓN ON-THE-FLY DE 'a + b * 5'
PILA: [0: $]

---> DESPLAZAR (Shift): Token 'id' (Atributo: a) -> Estado 3
PILA: [0: $] [3: id, attr=a]

<--- REDUCIR (Reduce R5): F -> (1 símbolos)
   Atributos desapilados ($1..$1): ['a']
   Resultado sintetizado ($$ = a) apilado en estado 2
PILA: [0: $] [2: F, attr=a]

<--- REDUCIR (Reduce R4): T -> (1 símbolos)
   Atributos desapilados ($1..$1): ['a']
   Resultado sintetizado ($$ = a) apilado en estado 1
PILA: [0: $] [1: T, attr=a]

<--- REDUCIR (Reduce R2): E -> (1 símbolos)
   Atributos desapilados ($1..$1): ['a']
   Resultado sintetizado ($$ = a) apilado en estado 4
PILA: [0: $] [4: E, attr=a]

---> DESPLAZAR (Shift): Token '+' (Atributo: None) -> Estado 6
PILA: [0: $] [4: E, attr=a] [6: +]

---> DESPLAZAR (Shift): Token 'id' (Atributo: b) -> Estado 3
PILA: [0: $] [4: E, attr=a] [6: +] [3: id, attr=b]

<--- REDUCIR (Reduce R5): F -> (1 símbolos)
   Atributos desapilados ($1..$1): ['b']
   Resultado sintetizad

## Simulación Avanzada de una Máquina de Pila Abstracta (Stack Machine)  
En esta sección implementaremos un intérprete/evaluador para una máquina de pila virtual, comprendiendo cómo las expresiones aritméticas compiladas se ejecutan a bajo nivel.  
## Objetivos de Aprendizaje    
Simular el comportamiento de una máquina de pila abstracta para la evaluación y generación de instrucciones de tipo **PUSH/POP/EVAL**.

### [CÓDIGO 1.2] Implementación del Emulador de Máquina de Pila  

In [8]:
class MaquinaDePilaAbstracta:

    def __init__(self):
        self.pila = []
        self.memoria = {}  # Simula la memoria de variables

    def ejecutar_instrucciones(self, programa):
        print("=== INICIO DE EJECUCIÓN EN MÁQUINA DE PILA ===")
        for instruccion in programa:
            partes = instruccion.split()
            op = partes[0]

            if op == "PUSH_NUM":
                val = float(partes[1])
                self.pila.append(val)
                print(f"PUSH_NUM {val:<5} -> Pila: {self.pila}")

            elif op == "LOAD_VAR":
                var_nombre = partes[1]
                val = self.memoria.get(var_nombre, 0.0)
                self.pila.append(val)
                print(f"LOAD_VAR {var_nombre:<5} ({val}) -> Pila: {self.pila}")

            elif op == "STORE_VAR":
                var_nombre = partes[1]
                val = self.pila.pop()
                self.memoria[var_nombre] = val
                print(f"STORE_VAR {var_nombre:<4} ({val}) -> Pila: {self.pila}")

            elif op in ("ADD", "SUB", "MUL", "DIV"):
                b = self.pila.pop()
                a = self.pila.pop()
                res = 0.0
                if op == "ADD":
                    res = a + b
                elif op == "SUB":
                    res = a - b
                elif op == "MUL":
                    res = a * b
                elif op == "DIV":
                    res = a / b
                self.pila.append(res)
                print(f"{op:<12} ({a} {op} {b}) -> Pila: {self.pila}")


if __name__ == "__main__":
    # Ejemplo de programa traducido a instrucciones de Máquina de Pila:
    # Corresponde a la asignación: x = 5; y = 10; resultado = (x + y) * 2
    codigo_pila = [
        "PUSH_NUM 5",
        "STORE_VAR x",
        "PUSH_NUM 10",
        "STORE_VAR y",
        "LOAD_VAR x",
        "LOAD_VAR y",
        "ADD",
        "PUSH_NUM 2",
        "MUL",
        "STORE_VAR resultado",
    ]

    vm = MaquinaDePilaAbstracta()
    vm.ejecutar_instrucciones(codigo_pila)
    print("\nEstado final de la memoria principal:", vm.memoria)

=== INICIO DE EJECUCIÓN EN MÁQUINA DE PILA ===
PUSH_NUM 5.0   -> Pila: [5.0]
STORE_VAR x    (5.0) -> Pila: []
PUSH_NUM 10.0  -> Pila: [10.0]
STORE_VAR y    (10.0) -> Pila: []
LOAD_VAR x     (5.0) -> Pila: [5.0]
LOAD_VAR y     (10.0) -> Pila: [5.0, 10.0]
ADD          (5.0 ADD 10.0) -> Pila: [15.0]
PUSH_NUM 2.0   -> Pila: [15.0, 2.0]
MUL          (15.0 MUL 2.0) -> Pila: [30.0]
STORE_VAR resultado (30.0) -> Pila: []

Estado final de la memoria principal: {'x': 5.0, 'y': 10.0, 'resultado': 30.0}


## 4. Sección Práctica 2: Tabla de Símbolos Jerárquica (Gestión de Ámbitos / Scopes) y Verificador de Errores Semánticos

Para manejar estructuras de control o funciones en los lenguajes originales de los equipos, la Tabla de Símbolos debe gestionar niveles de anidamiento (Scopes).

### [CÓDIGO 2.1] Implementación de la Tabla deE Símbolos Jerárquica y verificación de errores semánticos   

In [9]:
# ==============================================================================
# PROGRAMACIÓN DE SISTEMAS DE BASE II
# IMPLEMENTACIÓN DE LA TABLA DE SÍMBOLOS JERÁRQUICA Y VERIFICADOR DE ERRORES SEMÁNTICOS
# Modalidad: POO Nativa Pura en Python (Sin librerías externas)
# ==============================================================================

class ErrorSemantico(Exception):
    """Clase base para excepciones semánticas personalizadas."""
    pass

class ErrorRedeclaracion(ErrorSemantico):
    """Excepción para cuando una variable ya fue declarada en el mismo ámbito."""
    pass

class ErrorVariableNoDeclarada(ErrorSemantico):
    """Excepción para cuando se intenta acceder a una variable inexistente en el contexto."""
    pass


class Simbolo:
    """Representa la información asociada a un identificador dentro de la Tabla de Símbolos."""
    def __init__(self, nombre, tipo_dato, categoria="variable", direccion_offset=0):
        self.nombre = nombre
        self.tipo_dato = tipo_dato          # ej. 'int', 'float', 'string'
        self.categoria = categoria          # ej. 'variable', 'parametro', 'constante'
        self.direccion_offset = direccion_offset  # Dirección de memoria relativa

    def __repr__(self):
        return f"Simbolo('{self.nombre}', tipo='{self.tipo_dato}', cat='{self.categoria}', offset={self.direccion_offset})"


class EntornoAmbito:
    """
    Representa un bloque o ámbito local específico dentro del programa.
    Mantiene un diccionario local de símbolos y una referencia a su entorno padre.
    """
    def __init__(self, id_bloque, padre=None, nivel_profundidad=0):
        self.id_bloque = id_bloque
        self.padre = padre                  # Entorno del bloque contenedor (padre)
        self.nivel_profundidad = nivel_profundidad
        self.tabla_local = {}               # Diccionario de identificadores locales

    def insertar_local(self, simbolo):
        """Inserta un símbolo en la tabla local si no existe previamente en este mismo bloque."""
        if simbolo.nombre in self.tabla_local:
            raise ErrorRedeclaracion(
                f"Error Semántico [Bloque {self.id_bloque}, Nivel {self.nivel_profundidad}]: "
                f"Redeclaración ilegal de la variable '{simbolo.nombre}' en el mismo ámbito local."
            )
        self.tabla_local[simbolo.nombre] = simbolo

    def buscar_local(self, nombre):
        """Busca un símbolo únicamente dentro del ámbito local actual."""
        return self.tabla_local.get(nombre, None)


class TablaSimbolosJerarquica:
    """
    Administrador central de la Tabla de Símbolos Jerárquica con soporte
    para bloques anidados y pila de entornos.
    """
    def __init__(self):
        self.contador_bloques = 0
        # Ámbito global inicial (Bloque 0, Nivel 0)
        self.ambito_global = EntornoAmbito(id_bloque=0, padre=None, nivel_profundidad=0)
        self.ambito_actual = self.ambito_global
        self.offset_acumulado = 0  # Gestor de direcciones relativas de memoria

    def abrir_bloque(self, nombre_contexto=""):
        """Abre un nuevo bloque anidado y actualiza el entorno activo."""
        self.contador_bloques += 1
        nuevo_nivel = self.ambito_actual.nivel_profundidad + 1
        nuevo_entorno = EntornoAmbito(
            id_bloque=self.contador_bloques,
            padre=self.ambito_actual,
            nivel_profundidad=nuevo_nivel
        )
        self.ambito_actual = nuevo_entorno
        tag = f" ({nombre_contexto})" if nombre_contexto else ""
        print(f"\n[+] ABRIR BLOQUE {self.contador_bloques}{tag} | Nivel de profundidad: {nuevo_nivel}")

    def cerrar_bloque(self):
        """Cierra el bloque actual y retorna al entorno padre."""
        if self.ambito_actual.padre is None:
            raise ErrorSemantico("Error: Intentando cerrar el ámbito global del programa.")

        print(f"\n[-] CERRAR BLOQUE {self.ambito_actual.id_bloque} | Retornando a Nivel {self.ambito_actual.padre.nivel_profundidad}")
        self.ambito_actual = self.ambito_actual.padre

    def insertar(self, nombre, tipo_dato, categoria="variable", tamano_bytes=4):
        """Inserta un nuevo símbolo en el ámbito local actual y calcula su dirección offset."""
        simbolo = Simbolo(nombre, tipo_dato, categoria, self.offset_acumulado)
        self.ambito_actual.insertar_local(simbolo)
        self.offset_acumulado += tamano_bytes
        print(f"   ✓ SIMBOLO REGISTRADO: '{nombre}' -> {simbolo} (Ámbito {self.ambito_actual.id_bloque})")
        return simbolo

    def buscar(self, nombre):
        """Busca un símbolo desde el ámbito actual ascendiendo por la jerarquía de padres (Scoping Léxico)."""
        entorno_cursor = self.ambito_actual
        while entorno_cursor is not None:
            simbolo = entorno_cursor.buscar_local(nombre)
            if simbolo is not None:
                print(f"   🔍 BUSQUEDA EXITOSA: '{nombre}' encontrado en Bloque {entorno_cursor.id_bloque} (Nivel {entorno_cursor.nivel_profundidad}) -> {simbolo}")
                return simbolo
            entorno_cursor = entorno_cursor.padre

        raise ErrorVariableNoDeclarada(
            f"Error Semántico [Bloque {self.ambito_actual.id_bloque}]: "
            f"La variable '{nombre}' no ha sido declarada en el ámbito activo ni en sus ámbitos contenedores."
        )

    def mostrar_cadena_ambitos(self):
        """Muestra los bloques actualmente activos desde la cima local hasta el ámbito global."""
        print("\n" + "=" * 70)
        print("ESTADO ACTUAL DE LA CADENA DE ÁMBITOS VISIBLES:")
        print("=" * 70)
        entorno_cursor = self.ambito_actual
        while entorno_cursor is not None:
            simbolos_str = ", ".join(entorno_cursor.tabla_local.keys()) if entorno_cursor.tabla_local else "vacío"
            print(f" ► [Bloque {entorno_cursor.id_bloque} | Nivel {entorno_cursor.nivel_profundidad}] Símbolos locales: {{{simbolos_str}}}")
            entorno_cursor = entorno_cursor.padre
        print("=" * 70 + "\n")


# ==============================================================================
# ENTORNO DE DEMOSTRACIÓN Y CASOS DE PRUEBA
# ==============================================================================
if __name__ == "__main__":
    ts = TablaSimbolosJerarquica()

    print("=" * 70)
    print("DEMOSTRACIÓN: TABLA DE SÍMBOLOS CON GESTIÓN DE ÁMBITOS Y ANIDAMIENTO")
    print("=" * 70)

    # 1. Declaraciones en Ámbito Global (Bloque 0)
    print("\n--- 1. Declaraciones en Ámbito Global ---")
    ts.insertar("global_x", "int")
    ts.insertar("PI", "float", categoria="constante")
    ts.mostrar_cadena_ambitos()

    # 2. Entrada a Función principal (Bloque 1)
    ts.abrir_bloque("Funcion main")
    ts.insertar("a", "int")
    ts.insertar("b", "float")

    # Búsqueda de variable global desde dentro de main
    ts.buscar("global_x")
    ts.mostrar_cadena_ambitos()

    # 3. Entrada a Bloque Interno (Bloque 2 - IF / WHILE)
    ts.abrir_bloque("Bloque interno IF")
    ts.insertar("a", "string")  # Ocultamiento/Sombra de variable 'a' local de main
    ts.insertar("temp", "int")

    ts.mostrar_cadena_ambitos()

    # Búsquedas en ámbito anidado
    ts.buscar("a")         # Encuentra la 'a' local del Bloque 2 (string)
    ts.buscar("b")         # Encuentra la 'b' del Bloque 1 (float)
    ts.buscar("global_x")  # Encuentra 'global_x' del Bloque 0 (int)

    # 4. Prueba de Manejo de Errores Semánticos
    print("\n--- 4. Demostración de Captura de Errores Semánticos ---")

    # Error A: Redeclaración en el mismo ámbito local
    try:
        print("\n   [Prueba A] Intentando redeclarar 'temp' en el mismo bloque local...")
        ts.insertar("temp", "float")
    except ErrorRedeclaracion as e:
        print(f"     [CAPTURADO EXITO] {e}")

    # 5. Salida del Bloque Interno
    ts.cerrar_bloque()

    # Tras cerrar Bloque 2, 'temp' ya no es visible y 'a' vuelve a ser la de main (int)
    ts.buscar("a")  # Ahora encuentra la 'a' de main (int)

    # Error B: Búsqueda de variable no declarada / fuera de ámbito
    try:
        print("\n   [Prueba B] Intentando buscar 'temp' fuera de su ámbito...")
        ts.buscar("temp")
    except ErrorVariableNoDeclarada as e:
        print(f"     [CAPTURADO EXITO] {e}")

    # Salida final de la función main
    ts.cerrar_bloque()
    ts.mostrar_cadena_ambitos()

DEMOSTRACIÓN: TABLA DE SÍMBOLOS CON GESTIÓN DE ÁMBITOS Y ANIDAMIENTO

--- 1. Declaraciones en Ámbito Global ---
   ✓ SIMBOLO REGISTRADO: 'global_x' -> Simbolo('global_x', tipo='int', cat='variable', offset=0) (Ámbito 0)
   ✓ SIMBOLO REGISTRADO: 'PI' -> Simbolo('PI', tipo='float', cat='constante', offset=4) (Ámbito 0)

ESTADO ACTUAL DE LA CADENA DE ÁMBITOS VISIBLES:
 ► [Bloque 0 | Nivel 0] Símbolos locales: {global_x, PI}


[+] ABRIR BLOQUE 1 (Funcion main) | Nivel de profundidad: 1
   ✓ SIMBOLO REGISTRADO: 'a' -> Simbolo('a', tipo='int', cat='variable', offset=8) (Ámbito 1)
   ✓ SIMBOLO REGISTRADO: 'b' -> Simbolo('b', tipo='float', cat='variable', offset=12) (Ámbito 1)
   🔍 BUSQUEDA EXITOSA: 'global_x' encontrado en Bloque 0 (Nivel 0) -> Simbolo('global_x', tipo='int', cat='variable', offset=0)

ESTADO ACTUAL DE LA CADENA DE ÁMBITOS VISIBLES:
 ► [Bloque 1 | Nivel 1] Símbolos locales: {a, b}
 ► [Bloque 0 | Nivel 0] Símbolos locales: {global_x, PI}


[+] ABRIR BLOQUE 2 (Bloque interno IF

## 5. Sección Práctica 3:  Integración del módulo de la tabla de símbolos con una clase de Árbol de Sintaxis Abstracta (AST)

En esta sección implementaremos el código para integrar la Tabla de Símbolos con una clase del Árbol de Sintaxis Abstracta (AST) para realizar una validación semántica completa de asignaciones y expresiones.

### [CÓDIGO 5.1] Integración de la Tabla deE Símboloscon AST.

In [10]:
# ==============================================================================
# PROGRAMACIÓN DE SISTEMAS DE BASE II
# PRÁCTICA INTEGRADORA: ÁRBOLES DE SINTAXIS ABSTRACTA (AST) + TABLA DE SÍMBOLOS
# Modalidad: POO Nativa Pura en Python (Sin librerías externas)
# ==============================================================================

class ErrorSemantico(Exception):
  """Excepción base para errores semánticos."""

  pass


class ErrorRedeclaracion(ErrorSemantico):
  """Error al intentar redeclarar una variable en el mismo ámbito."""

  pass


class ErrorVariableNoDeclarada(ErrorSemantico):
  """Error al intentar usar una variable no existente en el contexto."""

  pass


class ErrorTipoIncompatible(ErrorSemantico):
  """Error al operar o asignar tipos de datos incompatibles."""

  pass


# ------------------------------------------------------------------------------
# 1. MÓDULO DE LA TABLA DE SÍMBOLOS Y ÁMBITOS
# ------------------------------------------------------------------------------
class Simbolo:

  def __init__(
      self, nombre, tipo_dato, valor=None, categoria="variable", offset=0
  ):
    self.nombre = nombre
    self.tipo_dato = tipo_dato  # 'int', 'float', 'string'
    self.valor = valor
    self.categoria = categoria
    self.offset = offset

  def __repr__(self):
    return (
        f"Simbolo('{self.nombre}', tipo='{self.tipo_dato}', val={self.valor},"
        f" offset={self.offset})"
    )


class EntornoAmbito:

  def __init__(self, id_bloque, padre=None, nivel=0):
    self.id_bloque = id_bloque
    self.padre = padre
    self.nivel = nivel
    self.tabla_local = {}

  def insertar_local(self, simbolo):
    if simbolo.nombre in self.tabla_local:
      raise ErrorRedeclaracion(
          f"Error Semántico [Bloque {self.id_bloque}]: Redeclaración ilegal de"
          f" la variable '{simbolo.nombre}' en el mismo ámbito."
      )
    self.tabla_local[simbolo.nombre] = simbolo

  def buscar_local(self, nombre):
    return self.tabla_local.get(nombre, None)


class TablaSimbolosJerarquica:

  def __init__(self):
    self.contador_bloques = 0
    self.ambito_global = EntornoAmbito(id_bloque=0, padre=None, nivel=0)
    self.ambito_actual = self.ambito_global
    self.offset_acumulado = 0

  def abrir_bloque(self, etiqueta=""):
    self.contador_bloques += 1
    nuevo_entorno = EntornoAmbito(
        id_bloque=self.contador_bloques,
        padre=self.ambito_actual,
        nivel=self.ambito_actual.nivel + 1,
    )
    self.ambito_actual = nuevo_entorno
    tag = f" ({etiqueta})" if etiqueta else ""
    print(
        f"\n[+] ABRIR BLOQUE {self.contador_bloques}{tag} | Nivel"
        f" {nuevo_entorno.nivel}"
    )

  def cerrar_bloque(self):
    if self.ambito_actual.padre is None:
      raise ErrorSemantico("No se puede cerrar el ámbito global.")
    print(
        f"[-] CERRAR BLOQUE {self.ambito_actual.id_bloque} | Retornando a"
        f" Nivel {self.ambito_actual.padre.nivel}"
    )
    self.ambito_actual = self.ambito_actual.padre

  def declarar(self, nombre, tipo_dato, valor_inicial=None, tamano=4):
    simbolo = Simbolo(nombre, tipo_dato, valor_inicial, offset=self.offset_acumulado)
    self.ambito_actual.insertar_local(simbolo)
    self.offset_acumulado += tamano
    print(
        f"   ✓ DECLARACIÓN: Variable '{nombre}' ({tipo_dato}) registrada en"
        f" Bloque {self.ambito_actual.id_bloque}"
    )
    return simbolo

  def buscar(self, nombre):
    cursor = self.ambito_actual
    while cursor is not None:
      simbolo = cursor.buscar_local(nombre)
      if simbolo is not None:
        return simbolo
      cursor = cursor.padre
    raise ErrorVariableNoDeclarada(
        f"Error Semántico [Bloque {self.ambito_actual.id_bloque}]: La variable"
        f" '{nombre}' no ha sido declarada en el ámbito activo ni en sus"
        " contenedores."
    )


# ------------------------------------------------------------------------------
# 2. MÓDULO DEL ÁRBOL DE SINTAXIS ABSTRACTA (AST) CON VALIDACIÓN SEMÁNTICA
# ------------------------------------------------------------------------------
class NodoAST:

  def evaluar(self, ts):
    raise NotImplementedError


class Numero(NodoAST):

  def __init__(self, valor):
    self.valor = valor
    self.tipo_dato = "int" if isinstance(valor, int) else "float"

  def evaluar(self, ts):
    return self.valor, self.tipo_dato


class Variable(NodoAST):

  def __init__(self, nombre):
    self.nombre = nombre

  def evaluar(self, ts):
    simbolo = ts.buscar(self.nombre)
    if simbolo.valor is None:
      raise ErrorSemantico(
          f"Error Semántico: La variable '{self.nombre}' no ha sido"
          " inicializada."
      )
    return simbolo.valor, simbolo.tipo_dato


class Operador(NodoAST):

  def __init__(self, operador, izq, der):
    self.operador = operador
    self.izq = izq
    self.der = der

  def evaluar(self, ts):
    val_izq, tipo_izq = self.izq.evaluar(ts)
    val_der, tipo_der = self.der.evaluar(ts)

    if tipo_izq == "string" or tipo_der == "string":
      raise ErrorTipoIncompatible(
          f"Error Semántico de Tipos: Operación '{self.operador}' no permitida"
          f" entre '{tipo_izq}' y '{tipo_der}'."
      )

    tipo_resultante = (
        "float" if (tipo_izq == "float" or tipo_der == "float") else "int"
    )

    if self.operador == "+":
      res = val_izq + val_der
    elif self.operador == "-":
      res = val_izq - val_der
    elif self.operador == "*":
      res = val_izq * val_der
    elif self.operador == "/":
      if val_der == 0:
        raise ZeroDivisionError(
            "Error Semántico/Aritmético: División entre cero."
        )
      res = val_izq / val_der
      tipo_resultante = "float"

    return res, tipo_resultante


class Declaracion(NodoAST):

  def __init__(self, nombre, tipo_declarado, expresion_inicial=None):
    self.nombre = nombre
    self.tipo_declarado = tipo_declarado
    self.expresion_inicial = expresion_inicial

  def evaluar(self, ts):
    val_expr = None
    if self.expresion_inicial is not None:
      val_expr, tipo_expr = self.expresion_inicial.evaluar(ts)
      if self.tipo_declarado == "int" and tipo_expr == "string":
        raise ErrorTipoIncompatible(
            "Error Semántico de Tipos: No se puede inicializar variable 'int'"
            " con 'string'."
        )
    simbolo = ts.declarar(self.nombre, self.tipo_declarado, val_expr)
    return val_expr, self.tipo_declarado


class Asignacion(NodoAST):

  def __init__(self, nombre_var, expresion):
    self.nombre_var = nombre_var
    self.expresion = expresion

  def evaluar(self, ts):
    simbolo = ts.buscar(self.nombre_var)
    val_expr, tipo_expr = self.expresion.evaluar(ts)

    if simbolo.tipo_dato == "int" and tipo_expr == "string":
      raise ErrorTipoIncompatible(
          f"Error Semántico de Tipos: No se puede asignar 'string' a la"
          f" variable '{self.nombre_var}' de tipo '{simbolo.tipo_dato}'."
      )
    elif simbolo.tipo_dato == "int" and tipo_expr == "float":
      print(
          "   [COERCIÓN ADVERTENCIA]: Truncando 'float' a 'int' para asignación"
          f" en '{self.nombre_var}'"
      )
      val_expr = int(val_expr)

    simbolo.valor = val_expr
    print(
        f"   ✓ ASIGNACIÓN EXITOSA: {self.nombre_var} = {val_expr} (Tipo:"
        f" {simbolo.tipo_dato})"
    )
    return val_expr, simbolo.tipo_dato


class BloqueAST(NodoAST):

  def __init__(self, instrucciones, etiqueta=""):
    self.instrucciones = instrucciones
    self.etiqueta = etiqueta

  def evaluar(self, ts):
    ts.abrir_bloque(self.etiqueta)
    ultimo_res = None
    for instr in self.instrucciones:
      ultimo_res = instr.evaluar(ts)
    ts.cerrar_bloque()
    return ultimo_res


# ------------------------------------------------------------------------------
# 3. DEMOSTRACIÓN DE EJECUCIÓN
# ------------------------------------------------------------------------------
if __name__ == "__main__":
  ts = TablaSimbolosJerarquica()

  # Variables globales
  ts.declarar("global_a", "int", 10)
  ts.declarar("global_b", "float", 20.5)

  # AST de un programa con bloques anidados
  programa = BloqueAST(
      [
          Declaracion(
              "x", "int", Operador("*", Variable("global_a"), Numero(2))
          ),
          Asignacion("x", Operador("+", Variable("x"), Numero(5))),
          BloqueAST(
              [Declaracion("x", "float", Numero(3.14))],
              etiqueta="Bloque Anidado",
          ),
      ],
      etiqueta="Programa Principal",
  )

  print(
      "--- Ejecutando Evaluación del AST e Integración con Tabla de Símbolos"
      " ---"
  )
  programa.evaluar(ts)

  print("\n--- Pruebas de Captura de Errores Semánticos ---")
  # Captura de error: Operación int + string
  try:
    ts.declarar("texto", "string", "hola")
    err = Operador("+", Variable("global_a"), Variable("texto"))
    err.evaluar(ts)
  except ErrorTipoIncompatible as e:
    print(f"   [CAPTURADO ÉXITO]: {e}")

   ✓ DECLARACIÓN: Variable 'global_a' (int) registrada en Bloque 0
   ✓ DECLARACIÓN: Variable 'global_b' (float) registrada en Bloque 0
--- Ejecutando Evaluación del AST e Integración con Tabla de Símbolos ---

[+] ABRIR BLOQUE 1 (Programa Principal) | Nivel 1
   ✓ DECLARACIÓN: Variable 'x' (int) registrada en Bloque 1
   ✓ ASIGNACIÓN EXITOSA: x = 25 (Tipo: int)

[+] ABRIR BLOQUE 2 (Bloque Anidado) | Nivel 2
   ✓ DECLARACIÓN: Variable 'x' (float) registrada en Bloque 2
[-] CERRAR BLOQUE 2 | Retornando a Nivel 1
[-] CERRAR BLOQUE 1 | Retornando a Nivel 0

--- Pruebas de Captura de Errores Semánticos ---
   ✓ DECLARACIÓN: Variable 'texto' (string) registrada en Bloque 0
   [CAPTURADO ÉXITO]: Error Semántico de Tipos: Operación '+' no permitida entre 'int' y 'string'.
